In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import csv

from scipy.ndimage import label, binary_dilation, binary_opening
from skimage.measure import regionprops
from skimage.morphology import rectangle

# =========================
# Utilities
# =========================

def detect_background_rgb(arr):
    h, w, _ = arr.shape
    corners = np.array([
        arr[0, 0],
        arr[0, w - 1],
        arr[h - 1, 0],
        arr[h - 1, w - 1],
    ])
    uniq, counts = np.unique(corners.reshape(-1, 3), axis=0, return_counts=True)
    return uniq[np.argmax(counts)]

def color_distance(arr, bg):
    return np.linalg.norm(arr.astype(float) - bg.astype(float), axis=2)

def save_region(img, bbox, out_path):
    x0, y0, x1, y1 = bbox
    img.crop((x0, y0, x1, y1)).save(out_path)

# =========================
# Phase-1a Core
# =========================

def extract_regions(image_path: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)
    regions_dir = output_dir / "regions"
    regions_dir.mkdir(exist_ok=True)

    img = Image.open(image_path).convert("RGB")
    arr = np.array(img)

    bg = detect_background_rgb(arr)
    fg = color_distance(arr, bg) > 40

    results = []

    # =====================================================
    # PASS A — TEXT REGIONS (merge glyphs)
    # =====================================================
    text_mask = fg.copy()

    # Dilate horizontally to merge letters into words
    text_mask = binary_dilation(
        text_mask,
        structure=rectangle(1, 5)
    )

    labeled_text, _ = label(text_mask)

    for r in regionprops(labeled_text):
        y0, x0, y1, x1 = r.bbox
        h = y1 - y0
        w = x1 - x0

        if h < 12 or w < 12:
            continue
        if h > 80:
            continue  # not text

        region_id = f"text_{len(results):04d}"
        out = regions_dir / f"{region_id}.png"

        save_region(img, (x0, y0, x1, y1), out)

        results.append({
            "region_id": region_id,
            "type": "text",
            "bbox": [x0, y0, w, h],
            "pixel_count": int(r.area)
        })

    # =====================================================
    # PASS B — BAR REGIONS (vertical rectangles)
    # =====================================================
    bar_mask = fg.copy()

    # Remove thin structures (gridlines)
    bar_mask = binary_opening(
        bar_mask,
        structure=rectangle(5, 1)
    )

    labeled_bars, _ = label(bar_mask)

    for r in regionprops(labeled_bars):
        y0, x0, y1, x1 = r.bbox
        h = y1 - y0
        w = x1 - x0

        if h < 60 or w < 15:
            continue

        aspect = h / max(w, 1)
        if aspect < 1.5:
            continue  # bars are tall

        region_id = f"bar_{len(results):04d}"
        out = regions_dir / f"{region_id}.png"

        save_region(img, (x0, y0, x1, y1), out)

        results.append({
            "region_id": region_id,
            "type": "bar",
            "bbox": [x0, y0, w, h],
            "pixel_count": int(r.area)
        })

    # =====================================================
    # PASS C — LINE REGIONS (gridlines / axes)
    # =====================================================
    line_mask = fg.copy()

    line_mask = binary_opening(
        line_mask,
        structure=rectangle(1, 7)
    )

    labeled_lines, _ = label(line_mask)

    for r in regionprops(labeled_lines):
        y0, x0, y1, x1 = r.bbox
        h = y1 - y0
        w = x1 - x0

        if h <= 3 and w > 40 or w <= 3 and h > 40:
            region_id = f"line_{len(results):04d}"
            out = regions_dir / f"{region_id}.png"

            save_region(img, (x0, y0, x1, y1), out)

            results.append({
                "region_id": region_id,
                "type": "line",
                "bbox": [x0, y0, w, h],
                "pixel_count": int(r.area)
            })

    # =====================================================
    # CSV OUTPUT
    # =====================================================
    with open(output_dir / "regions.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["region_id", "type", "bbox", "pixel_count"]
        )
        writer.writeheader()
        for r in results:
            writer.writerow(r)

    return results

# =========================
# Entry Point
# =========================

if __name__ == "__main__":
    extract_regions(
        image_path=Path("input_images/Figure_01.png"),
        output_dir=Path("output_phase1a/Figure_01")
    )


In [2]:
run_phase2a(
    image_dir="docs/images",
    output_root="output_phase2a"
)


c:\Users\scien\miniconda3\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[{'figure_id': 'Picture1',
  'background_rgb': [255, 255, 255],
  'region_count': 139,
  'output_dir': 'output_phase2a\\Picture1'}]